# Loan Default Prediction

**Goal:** predict whether a loan will default (`Status` = 1) or not (`Status` = 0).

**Pipeline (same order as the churn preprocessing script):**
1. Load and explore the data
2. Clean the data (drop ID / useless / leaky columns, remove duplicates)
3. **Split first** into train and test
4. **Correlation matrix** and removal of highly correlated features
5. Median imputation, scaling, one-hot encoding (fitted on train only)
6. SMOTE for class balance (train only)
7. Train 4 models and compare them (metrics, confusion matrices, ROC curves)

**How to run in VS Code:** install the *Python* and *Jupyter* extensions, open this file, select your Python kernel (top right), then click **Run All**.

## 1. Install and import libraries
Run the install line once (remove the `#`), then restart the kernel if needed.

In [ ]:
# %pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn joblib

In [ ]:
import os
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay,
                             classification_report)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 2. Load the data
Change `DATA_PATH` if your file is somewhere else.

In [ ]:
DATA_PATH = r'C:\Users\user\Downloads\Loan_Default.csv (1)\Loan_Default.csv'

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

## 3. Explore the data

In [ ]:
df.info()

In [ ]:
# Missing values (only columns that have some)
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

pd.DataFrame({
    'missing': missing,
    'percent': (missing / len(df) * 100).round(2)
})

In [ ]:
# Target distribution
print(df['Status'].value_counts())
print()
print(df['Status'].value_counts(normalize=True).round(3))

sns.countplot(x='Status', data=df)
plt.title('Target distribution (0 = No default, 1 = Default)')
plt.show()

### Leakage check
Some columns are missing **mostly for loans that defaulted**. If that is true, the missing pattern itself
tells the model the answer, which gives an unrealistic accuracy near 100%.
Look at the share of missing values for each `Status`:

In [ ]:
leaky_columns = ['rate_of_interest', 'Interest_rate_spread', 'Upfront_charges']

# fraction of missing values for Status = 0 and Status = 1
df[leaky_columns].isnull().groupby(df['Status']).mean().round(3)

## 4. Data cleaning
- Drop `ID` (just an identifier) and `year` (same value for every row).
- Drop the leaky columns found above (set `DROP_LEAKY = False` to keep them and compare).
- Remove duplicate rows.

In [ ]:
DROP_LEAKY = True

df = df.drop(columns=['ID', 'year'])

if DROP_LEAKY:
    df = df.drop(columns=leaky_columns)

before = len(df)
df = df.drop_duplicates()
print("Duplicate rows removed:", before - len(df))

X = df.drop('Status', axis=1)
y = df['Status']
print("X shape:", X.shape, "| y shape:", y.shape)

## 5. Train-test split
The split happens **before** any imputing, scaling or encoding, so no information from the test set leaks into training.
`stratify=y` keeps the same default ratio in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("\nTrain class ratio:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest class ratio:\n", y_test.value_counts(normalize=True).round(3))

## 6. Correlation matrix
Calculated on the **training data only**. It shows which numeric features move together.
If two features are almost the same (correlation above the threshold), we keep one and drop the other.

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric columns    :", num_cols)
print("Categorical columns:", cat_cols)

In [ ]:
# Correlation matrix of numeric features together with the target
corr_matrix = X_train[num_cols].join(y_train).corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix (training data)')
plt.show()

In [ ]:
# Correlation of each numeric feature with the target
corr_matrix['Status'].drop('Status').sort_values().plot(kind='barh', figsize=(7, 4))
plt.title('Correlation with Status')
plt.show()

In [ ]:
# Drop one feature from every highly correlated pair
THRESHOLD = 0.85

abs_corr = X_train[num_cols].corr().abs()
upper = abs_corr.where(np.triu(np.ones(abs_corr.shape), k=1).astype(bool))
high_corr = [col for col in upper.columns if (upper[col] > THRESHOLD).any()]

print(f"Features with correlation above {THRESHOLD}:", high_corr)

X_train = X_train.drop(columns=high_corr)
X_test = X_test.drop(columns=high_corr)
num_cols = [c for c in num_cols if c not in high_corr]

print("Numeric columns kept:", num_cols)

## 7. Impute, scale and encode
Everything is **fitted on the training data only** and then applied to the test data.
- Numeric: median imputation, then `StandardScaler`
- Categorical: missing values become the label `'Missing'`, then `OneHotEncoder`

In [ ]:
# --- Numeric ---
imputer = SimpleImputer(strategy='median')
x_train_num = imputer.fit_transform(X_train[num_cols])
x_test_num = imputer.transform(X_test[num_cols])

scaler = StandardScaler()
x_train_num = scaler.fit_transform(x_train_num)
x_test_num = scaler.transform(x_test_num)

# --- Categorical ---
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
x_train_cat = ohe.fit_transform(X_train[cat_cols].fillna('Missing'))
x_test_cat = ohe.transform(X_test[cat_cols].fillna('Missing'))

# --- Combine ---
X_train_final = np.hstack((x_train_num, x_train_cat))
X_test_final = np.hstack((x_test_num, x_test_cat))

feature_names = num_cols + list(ohe.get_feature_names_out(cat_cols))

print("X_train_final:", X_train_final.shape)
print("X_test_final :", X_test_final.shape)
print("Any NaN left in train:", np.isnan(X_train_final).any())

## 8. Balance the classes with SMOTE
SMOTE creates synthetic examples of the minority class. It is applied to the **training data only**.
The test set stays untouched, so the results reflect real-world data.

In [ ]:
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_final, y_train)

print("Before SMOTE:")
print(y_train.value_counts())
print("\nAfter SMOTE:")
print(y_train_bal.value_counts())

"9. Train the models"

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=12, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    'Gradient Boosting': HistGradientBoostingClassifier(random_state=42),
}

results = []
predictions = {}
probabilities = {}
fitted_models = {}

for name, model in models.items():
    start = time.time()
    model.fit(X_train_bal, y_train_bal)

    y_pred = model.predict(X_test_final)
    y_prob = model.predict_proba(X_test_final)[:, 1]

    predictions[name] = y_pred
    probabilities[name] = y_prob
    fitted_models[name] = model

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'Train time (s)': round(time.time() - start, 1),
    })
    print(f"{name} done")

## 10. Model comparison
- **Accuracy**: share of all predictions that are correct
- **Precision**: of the loans predicted as default, how many really defaulted
- **Recall**: of the loans that really defaulted, how many we caught
- **F1-score**: balance of precision and recall
- **ROC-AUC**: how well the model separates the two classes (1.0 is perfect, 0.5 is random)

For loan default, recall and F1 matter more than accuracy, because missing a real defaulter is costly.

In [ ]:
results_df = pd.DataFrame(results).set_index('Model').sort_values('F1-score', ascending=False)
results_df.round(4)

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']

results_df[metrics].plot(kind='bar', figsize=(11, 5))
plt.title('Model comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.show()

## 11. Confusion matrices
Each square shows how many test loans fall into each case:
- top-left: correctly predicted **No default**
- bottom-right: correctly predicted **Default**
- top-right: false alarms (predicted default, actually fine)
- bottom-left: missed defaulters (the costly mistake)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for ax, (name, y_pred) in zip(axes.ravel(), predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['No Default', 'Default'],
        cmap='Blues', colorbar=False, ax=ax
    )
    ax.set_title(name)
    ax.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# Same matrices as percentages (each row adds up to 100%)
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for ax, (name, y_pred) in zip(axes.ravel(), predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, normalize='true',
        display_labels=['No Default', 'Default'],
        cmap='Greens', colorbar=False, values_format='.1%', ax=ax
    )
    ax.set_title(name + ' (normalized)')
    ax.grid(False)

plt.tight_layout()
plt.show()

## 12. ROC curves

In [ ]:
plt.figure(figsize=(7, 6))

for name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC curves')
plt.legend()
plt.show()

## 13. Best model
The best model is chosen by F1-score on the test set.

In [ ]:
best_name = results_df['F1-score'].idxmax()
print("Best model:", best_name)
print()
print(classification_report(y_test, predictions[best_name],
                            target_names=['No Default', 'Default']))

In [ ]:
# Most important features (from the Random Forest)
rf = fitted_models['Random Forest']

importances = pd.Series(rf.feature_importances_, index=feature_names)
importances.sort_values().tail(15).plot(kind='barh', figsize=(8, 6))
plt.title('Top 15 features (Random Forest)')
plt.xlabel('Importance')
plt.show()

**Sanity check:** if any model scores about 1.00 (100%), something is leaking the answer.
Check `DROP_LEAKY = True` in section 4 and look at the feature importance chart above.
A realistic result for this dataset is well below 1.0.

## 14. Save the best model and preprocessing objects

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(fitted_models[best_name], 'models/best_model.pkl')
joblib.dump(imputer, 'models/imputer.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(ohe, 'models/ohe.pkl')

print("Saved best model:", best_name)

## 15. Conclusion
Write your findings here, for example:
- Which model performed best and on which metric
- How many defaulters it caught (recall) and how many false alarms it raised
- Which features mattered most
- Why the leaky columns were removed